# Notebook 02 — Treinamento e Compressão de Modelos

## Aula 04: Otimização de Modelos — Pruning e Quantização

---

### 🎯 Objetivos deste Notebook

1. Treinar o modelo base (**SimpleCNN**) no CIFAR-10
2. Aplicar **pruning não-estruturado** baseado em magnitude (Han et al., 2015)
3. Aplicar **pruning estruturado** e comparar com não-estruturado
4. Realizar **fine-tuning** pós-pruning para recuperar acurácia
5. Aplicar **quantização dinâmica INT8** (Jacob et al., 2018)
6. Treinar um modelo **student** via **Knowledge Distillation** (Hinton et al., 2015)
7. Montar o **pipeline integrado de compressão** (Han et al., 2015)

### 📖 Conceito Teórico (Documento 04 — Saiba Mais)

> O pipeline moderno de compressão segue uma sequência refinada:
> 1. Treinamento do modelo completo até convergência (baseline)
> 2. Pruning iterativo com fine-tuning intercalado
> 3. Knowledge Distillation (opcional) para recuperar acurácia
> 4. Quantização — tipicamente a última etapa

> A fórmula de **Knowledge Distillation** (Hinton et al., 2015):
> $$\mathcal{L} = \alpha \cdot T^2 \cdot D_{KL}(p^T_{student} \| p^T_{teacher}) + (1 - \alpha) \cdot \mathcal{L}_{CE}(p_{student}, y)$$

---

In [ ]:
# Imports necessários
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn

# Adicionar diretório raiz ao path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_preprocessing import DataPreprocessor
from src.model import (
    SimpleCNN,
    StudentCNN,
    apply_unstructured_pruning,
    apply_structured_pruning,
    remove_pruning_reparametrization,
    compute_sparsity,
    quantize_dynamic,
    count_parameters,
    get_model_size_mb,
)
from src.training import (
    train_model,
    fine_tune,
    train_with_distillation,
    evaluate,
)
from src.evaluation import (
    calculate_accuracy,
    measure_latency,
    plot_training_history,
    plot_weight_distribution,
    plot_sparsity_distribution,
)
from src.utils import save_model, save_metrics, set_seed, get_device

# Configurações
plt.style.use("seaborn-v0_8-whitegrid")
set_seed(42)
device = get_device()
print(f"Dispositivo: {device}")

# Criar diretórios de saída
Path("outputs/models").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/logs").mkdir(parents=True, exist_ok=True)

## 1. Preparação dos Dados

In [ ]:
# Carregar CIFAR-10
preprocessor = DataPreprocessor(data_dir="data/raw", batch_size=128, seed=42)
preprocessor.load_data(download=True)
train_loader, val_loader, test_loader = preprocessor.split_data(val_fraction=0.1)
calibration_loader = preprocessor.get_calibration_loader(n_samples=1000)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

## 2. Treinamento do Modelo Base (Baseline)

Primeiro passo do pipeline de compressão (doc 04): treinar modelo denso até
convergência para estabelecer baseline de acurácia e métricas de referência.

> **Nota:** para fins pedagógicos usamos poucas épocas. Em produção,
> treinar até convergência completa.

In [ ]:
# Treinar modelo base (SimpleCNN)
# Reduzimos épocas para execução rápida — aumente para melhor acurácia
BASELINE_EPOCHS = 5

model_base = SimpleCNN(num_classes=10)
history_base = train_model(
    model_base,
    train_loader,
    val_loader,
    epochs=BASELINE_EPOCHS,
    lr=0.01,
    device=device,
)

# Salvar modelo base
save_model(model_base, "outputs/models/baseline.pt")

In [ ]:
# Visualizar curvas de treinamento
plot_training_history(
    history_base,
    title="SimpleCNN Baseline",
    save_path="outputs/figures/baseline_training.png",
)

In [ ]:
# Acurácia baseline no test set
baseline_acc = calculate_accuracy(model_base, test_loader, device)
baseline_latency = measure_latency(model_base, device=device)
baseline_size = get_model_size_mb(model_base)

print(f"Baseline — Acurácia: {baseline_acc:.4f}")
print(f"Baseline — Latência média: {baseline_latency['mean_ms']:.2f} ms")
print(f"Baseline — Tamanho: {baseline_size:.2f} MB")

## 3. Pruning Não-Estruturado (Magnitude-Based)

Implementa o paradigma de Han et al. (2015): remove pesos individuais com
menor magnitude absoluta. Conforme doc 04, Tabela 1, este tipo de pruning
atinge taxas de compressão de 80-95% com alta preservação de acurácia.

O critério magnitude-based assume que pesos com |w| < τ contribuem
minimamente para as ativações subsequentes.

In [ ]:
import copy

# Aplicar pruning L1 não-estruturado (30% dos pesos)
# Conforme Snippet 1 do doc 04 (seção Hands On)
model_pruned_30 = copy.deepcopy(model_base)
model_pruned_30 = apply_unstructured_pruning(model_pruned_30, amount=0.3, prune_type="l1")

sparsity_30 = compute_sparsity(model_pruned_30)
print("Esparsidade com 30% pruning:")
for layer, sp in sparsity_30.items():
    print(f"  {layer}: {sp:.1f}%")

In [ ]:
# Visualizar distribuição de pesos após pruning (pico em zero)
plot_weight_distribution(
    model_pruned_30,
    title="Distribuição de Pesos — Após Pruning 30%",
    save_path="outputs/figures/weight_dist_pruned30.png",
)

In [ ]:
# Esparsidade crescente: 30%, 50%, 70%, 90%
# Avaliar acurácia em cada nível (Lottery Ticket exploration)
pruning_results = {}
for amount in [0.3, 0.5, 0.7, 0.9]:
    m = copy.deepcopy(model_base)
    m = apply_unstructured_pruning(m, amount=amount)
    acc = calculate_accuracy(m, test_loader, device)
    sp = compute_sparsity(m)["global"]
    pruning_results[f"Pruned {int(amount*100)}%"] = {
        "accuracy": acc,
        "sparsity": sp,
    }
    print(f"Pruning {amount*100:.0f}% → Esparsidade: {sp:.1f}% | Acurácia: {acc:.4f}")

## 4. Pruning Estruturado

Conforme doc 04 (Tabela 1), pruning estruturado remove **estruturas completas**
(filtros em CNNs), resultando em aceleração real em GPU/CPU padrão.
Trade-off: taxa de compressão menor (30-70%) porém speedup significativo.

In [ ]:
# Pruning estruturado — 30% dos filtros
model_struct_pruned = copy.deepcopy(model_base)
model_struct_pruned = apply_structured_pruning(model_struct_pruned, amount=0.3)

sp_struct = compute_sparsity(model_struct_pruned)
acc_struct = calculate_accuracy(model_struct_pruned, test_loader, device)
print(f"Pruning Estruturado 30%: Esparsidade global = {sp_struct['global']:.1f}% | Acc = {acc_struct:.4f}")

# Visualizar esparsidade por camada
plot_sparsity_distribution(
    sp_struct,
    save_path="outputs/figures/structured_sparsity.png",
)

## 5. Fine-Tuning Pós-Pruning

Segundo passo do pipeline de Han et al. (2015): após pruning, fine-tune com
learning rate reduzida para que pesos remanescentes compensem conexões removidas.

In [ ]:
# Fine-tune do modelo com 50% pruning não-estruturado
model_ft = copy.deepcopy(model_base)
model_ft = apply_unstructured_pruning(model_ft, amount=0.5)

acc_before_ft = calculate_accuracy(model_ft, test_loader, device)
print(f"Antes do fine-tune: Acc = {acc_before_ft:.4f}")

history_ft = fine_tune(
    model_ft,
    train_loader,
    val_loader,
    epochs=3,
    lr=0.001,
    device=device,
)

acc_after_ft = calculate_accuracy(model_ft, test_loader, device)
print(f"Após fine-tune: Acc = {acc_after_ft:.4f} (recuperação: {acc_after_ft - acc_before_ft:+.4f})")

# Permanentizar pruning para exportação
model_ft = remove_pruning_reparametrization(model_ft)
save_model(model_ft, "outputs/models/pruned_finetuned.pt")

In [ ]:
plot_training_history(
    history_ft,
    title="Fine-Tune Pós-Pruning (50%)",
    save_path="outputs/figures/finetune_history.png",
)

## 6. Quantização Dinâmica INT8

Conforme discutido na seção *Saiba Mais* do doc 04, a quantização dinâmica
converte camadas Linear para INT8 em tempo de inferência. Pesos ficam
estáticos em INT8; ativações são quantizadas dinamicamente.

Fórmula de quantização uniforme (doc 04):
$$q = \text{round}\left(\frac{x - z}{s}\right), \quad \hat{x} = s \cdot q + z$$

In [ ]:
# Quantização dinâmica — Snippet 2 do doc 04
# Aplica sobre o modelo baseline (FP32)
model_quantized = quantize_dynamic(model_base)

size_q = get_model_size_mb(model_quantized)
acc_q = calculate_accuracy(model_quantized, test_loader, "cpu")
latency_q = measure_latency(model_quantized, device="cpu")

print(f"Modelo quantizado (INT8 dinâmico):")
print(f"  Tamanho: {size_q:.2f} MB (baseline: {baseline_size:.2f} MB)")
print(f"  Acurácia: {acc_q:.4f} (baseline: {baseline_acc:.4f})")
print(f"  Latência: {latency_q['mean_ms']:.2f} ms (baseline: {baseline_latency['mean_ms']:.2f} ms)")

## 7. Knowledge Distillation

Implementa o framework **teacher-student** de Hinton et al. (2015). O teacher
(SimpleCNN treinado) gera *soft labels* com temperatura T, transferindo
*dark knowledge* para o student (StudentCNN, ~4× menor).

Conforme doc 04:
> As saídas do teacher contêm mais informação que os rótulos hard originais.
> O *dark knowledge* revela relações entre classes (ex: gatos são mais
> similares a cachorros que a carros).

In [ ]:
# Treinar student via Knowledge Distillation
student = StudentCNN(num_classes=10)
print(f"Student: {count_parameters(student):,} params ({get_model_size_mb(student):.2f} MB)")

history_distill = train_with_distillation(
    student=student,
    teacher=model_base,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=5,
    lr=0.01,
    temperature=4.0,  # T > 1 suaviza distribuição (doc 04)
    alpha=0.7,        # Peso maior para soft loss
    device=device,
)

acc_student = calculate_accuracy(student, test_loader, device)
print(f"\nStudent distilled: Acc = {acc_student:.4f}")

save_model(student, "outputs/models/student_distilled.pt")

In [ ]:
plot_training_history(
    history_distill,
    title="Knowledge Distillation — StudentCNN",
    save_path="outputs/figures/distillation_history.png",
)

## 8. Treinar Student sem Distillation (comparação)

Para demonstrar o valor do *dark knowledge*, treinamos o mesmo student
diretamente com cross-entropy (sem teacher).

In [ ]:
# Student treinado diretamente (sem distillation)
student_direct = StudentCNN(num_classes=10)
history_direct = train_model(
    student_direct,
    train_loader,
    val_loader,
    epochs=5,
    lr=0.01,
    device=device,
)

acc_direct = calculate_accuracy(student_direct, test_loader, device)
print(f"Student direto (sem distillation): Acc = {acc_direct:.4f}")
print(f"Student distilled:                  Acc = {acc_student:.4f}")
print(f"Ganho com distillation:             {acc_student - acc_direct:+.4f}")

## 9. Pipeline Integrado de Compressão

Conforme Han et al. (2015) e Snippet 4 do doc 04: **Pruning → Fine-tune → Quantização**.
Este pipeline alcança compressões de 10-50× com < 2% perda de acurácia.

In [ ]:
# Pipeline: Pruning 50% → Fine-tune 3 épocas → Quantização INT8
model_pipeline = copy.deepcopy(model_base)

# Etapa 1: Pruning
print("=== Etapa 1: Pruning 50% ===")
model_pipeline = apply_unstructured_pruning(model_pipeline, amount=0.5)
acc_p = calculate_accuracy(model_pipeline, test_loader, device)
print(f"  Acc após pruning: {acc_p:.4f}")

# Etapa 2: Fine-tune
print("\n=== Etapa 2: Fine-tune ===")
_ = fine_tune(model_pipeline, train_loader, val_loader, epochs=3, lr=0.001, device=device)
acc_ft = calculate_accuracy(model_pipeline, test_loader, device)
print(f"  Acc após fine-tune: {acc_ft:.4f}")

# Permanentizar pruning
model_pipeline = remove_pruning_reparametrization(model_pipeline)

# Etapa 3: Quantização
print("\n=== Etapa 3: Quantização INT8 ===")
model_pipeline = quantize_dynamic(model_pipeline)
acc_final = calculate_accuracy(model_pipeline, test_loader, "cpu")
lat_final = measure_latency(model_pipeline, device="cpu")
size_final = get_model_size_mb(model_pipeline)

print(f"\n{'='*55}")
print(f"  PIPELINE COMPLETO")
print(f"{'='*55}")
print(f"  Acurácia:  {acc_final:.4f} (baseline: {baseline_acc:.4f})")
print(f"  Latência:  {lat_final['mean_ms']:.2f} ms")
print(f"  Tamanho:   {size_final:.2f} MB (baseline: {baseline_size:.2f} MB)")
print(f"  Compressão: {baseline_size/size_final:.1f}×")
print(f"{'='*55}")

# Salvar métricas
save_metrics(
    {
        "baseline_acc": baseline_acc,
        "pipeline_acc": acc_final,
        "baseline_size_mb": baseline_size,
        "pipeline_size_mb": size_final,
        "compression_ratio": baseline_size / size_final,
        "pipeline_latency_ms": lat_final["mean_ms"],
    },
    "outputs/logs/pipeline_metrics.json",
)

## 10. Resumo e Próximos Passos

Neste notebook implementamos o pipeline completo do documento 04:

| Técnica | Acurácia | Tamanho |
|---------|----------|---------|
| Baseline (FP32) | Referência | Referência |
| Pruning 50% + Fine-tune | Recuperada | ~50% dos pesos zero |
| Quantização INT8 | < 1% degradação | ~4× menor |
| Pipeline combinado | Trade-off controlado | Máxima compressão |
| Knowledge Distillation | Student com dark knowledge | Modelo ~4× menor |

### ➡️ Próximo Notebook: `03_avaliacao.ipynb`

No próximo notebook faremos avaliação detalhada com benchmarks de latência,
comparação entre todas variantes e análise de trade-offs conforme a Tabela 2
do documento 04 (Jacob et al., 2018; Blalock et al., 2020).